# VIVO Backgammon — Entrenamiento TD(0) en GPU (Colab)

Red wide-net (198→256→128→64→1, tanh), idéntica a la del navegador (`src/features/ai-worker/nn-model.ts`).
El motor de backgammon está portado 1:1 a Python (`bg_engine.py`) para que la red juegue por las MISMAS reglas.

**Salida:** `public/model_weights.json` en el formato EXACTO que el navegador carga → sin cambios en el frontend.

## Pasos
1. Sube esta carpeta (`colab/`) a Colab o clona el repo y abre este notebook con GPU habilitada (Entorno de ejecución → Cambiar tipo de entorno → GPU).
2. Ejecuta las celdas en orden.
3. Al terminar, descarga `public/model_weights.json` y colócalo en `public/` de tu proyecto VIVO (o súbelo al repo y haz pull).

## Verificación de colapso (sin TensorFlow, rápido)
La última celda hace un forward-pass en Python puro sobre los pesos exportados y reporta COLLAPSED / ALIVE. Úsala tras cada corrida.

In [ ]:
# === 0. Setup: instala dependencias y habilita GPU ===
import subprocess, sys, os, json

# TensorFlow con GPU (Colab ya trae CUDA 12 + cuDNN; pip install tf los usa)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow', 'numpy'])

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus)
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('GPU habilitada para entrenamiento.')
else:
    print('⚠️ NO se detectó GPU — el entrenamiento será lento (CPU).')

# Asegura carpeta de salida
os.makedirs('public', exist_ok=True)

In [ ]:
# === 1. Sube bg_engine.py y bg_net.py a este entorno de Colab ===
# Opción A (recomendada): clona el repo y copia las dos carpetas.
# Opción B: usa el panel lateral 'Archivos' -> Subir y sube ambos .py a /content.
#
# Si clonaste el repo:
#   !git clone https://github.com/Pirzl/Backgammon-AR-Pro.git
#   %cd Backgammon-AR-Pro/colab
#   os.makedirs('public', exist_ok=True)

# Verifica que bg_engine.py y bg_net.py estén en el cwd
assert os.path.exists('bg_engine.py'), 'Falta bg_engine.py en el directorio de trabajo'
assert os.path.exists('bg_net.py'), 'Falta bg_net.py en el directorio de trabajo'
print('Archivos listos.')

In [ ]:
# === 2. (Opcional) Reanuda desde pesos previos ===
# Si tienes un model_weights.json de una corrida anterior, súbelo y descomenta:
# RESUME = 'model_weights.json'
RESUME = None
print('RESUME =', RESUME)

In [ ]:
# === 3. Entrena (TD(0) vs heurístico, GPU) ===
# Empieza conservador y vigila el log. Si ves 'rate' subir >0.55 en eval, está aprendiendo.
# --games controla el total; para millones de partidas usa --games 2000000 con --save-every alto.
# El --stop-rate 0.60 autodetiene cuando el NN vence al heurístico >=60% dos evals seguidas.

import bg_net as net

cmd = [
    sys.executable, 'bg_net.py',
    '--games', '500000',
    '--opponent', 'heuristic',
    '--label', 'td0',
    '--exploration', '0.15',
    '--max-moves', '400',
    '--epochs', '3',
    '--eval-every', '250',
    '--eval-games', '200',
    '--save-every', '50',
    '--stop-rate', '0.60',
    '--stop-streak', '2',
    '--out', 'public/model_weights.json',
    '--seed', '1',
]
if RESUME:
    cmd += ['--weights', RESUME]

print('Arrancando entrenamiento GPU...')
subprocess.check_call(cmd)

In [ ]:
# === 4. Verificación de colapso (Python puro, sin TF) ===
# Lee public/model_weights.json y hace forward-pass sobre posiciones variadas.
# Si std < 1e-3 -> COLLAPSED (la red no reacciona). Si >50% unidades muertas -> WARNING.

import json, math, random
import bg_engine as bg

raw = json.load(open('public/model_weights.json'))
w = raw['weights']
W1,b1,W2,b2,W3,b3,W4,b4 = [l['data'] for l in w]

def mat(M, x, b):
    return [b[j] + sum(x[i]*M[i][j] for i in range(len(x))) for j in range(len(b))]

def fwd(fv):
    h1 = [max(0,v) for v in mat(W1, fv, b1)]
    h2 = [max(0,v) for v in mat(W2, h1, b2)]
    h3 = [max(0,v) for v in mat(W3, h2, b3)]
    y  = [math.tanh(v) for v in mat(W4, h3, b4)]
    return y[0], (h1,h2,h3)

rng = random.Random(1)
preds = []
hs = ([],[],[])
for t in ['white','black']:
    y,h = fwd(bg.encode_board(bg.INITIAL_BOARD, t)); preds.append(y)
    for k in range(8):
        b = [0]*30
        for _ in range(15): b[rng.randint(1,24)] += 1
        for _ in range(15): b[rng.randint(1,24)] -= 1
        y,h = fwd(bg.encode_board(b, t)); preds.append(y)
        for li,hh in enumerate(h): hs[li].append(hh)

mean = sum(preds)/len(preds)
var = sum((p-mean)**2 for p in preds)/len(preds)
std = math.sqrt(var)
print('preds=', [round(p,3) for p in preds])
print(f'mean={mean:.4f} std={std:.4f} min={min(preds):.3f} max={max(preds):.3f}')
dead = []
for li,st in enumerate(hs):
    n = len(st[0]); d = sum(1 for u in range(n) if all(abs(st[k][u])<1e-9 for k in range(len(st))))
    dead.append(d/n)
print('dead units:', [f'{x*100:.0f}%' for x in dead])

if std < 1e-3:
    print('VEREDICTO: COLLAPSED (salida no reacciona)')
elif any(x > 0.5 for x in dead):
    print('VEREDICTO: WARNING (>50% unidades muertas)')
else:
    print('VEREDICTO: ALIVE')

In [ ]:
# === 5. Descarga los pesos entrenados ===
# Doble clic en public/model_weights.json en el panel de archivos, o:
from google.colab import files
files.download('public/model_weights.json')
# Llévalo a: E:/Proyecto/BACKGAMMON/BACKGAMMON-VIVO - copia/public/model_weights.json